#Install Requirement

In [1]:
!pip install -q requests beautifulsoup4 lxml python-dateutil

##Library

In [2]:
import json
import time
import requests

from bs4 import BeautifulSoup
from dateutil import parser
from datetime import datetime
from urllib.parse import urljoin, urlparse

## Global Config

In [3]:
START_URL = "https://www.bisnis.com/"

MAX_PAGES = 5
REQUEST_DELAY = 1

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "Chrome/120.0 Safari/537.36"
    )
}

In [4]:
def get_soup(url):
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=30
        )

        response.raise_for_status()

        time.sleep(REQUEST_DELAY)

        return BeautifulSoup(
            response.text,
            "lxml"
        )

    except requests.RequestException as error:
        print(f"Request failed: {url}")
        print(f"Error: {error}")

        return None

## URL Validation

In [5]:
def normalize_url(url, base_url=START_URL):
    if not url:
        return None

    absolute_url = urljoin(
        base_url,
        url
    )

    absolute_url = absolute_url.split("?")[0]
    absolute_url = absolute_url.split("#")[0]

    return absolute_url.rstrip("/")

In [6]:
def is_bisnis_url(url):
    if not url:
        return False

    domain = urlparse(url).netloc.lower()

    return (
        domain == "bisnis.com"
        or domain.endswith(".bisnis.com")
    )

In [7]:
def is_article_url(url):
    return (
        is_bisnis_url(url)
        and "/read/" in url
    )

In [8]:
def parse_date(date_value):
    if not date_value:
        return None

    try:
        return parser.parse(
            str(date_value)
        )

    except (
        ValueError,
        TypeError,
        OverflowError
    ):
        return None

In [9]:
def parse_input_date(date_value):
    try:
        return datetime.strptime(
            date_value,
            "%Y-%m-%d"
        ).date()

    except ValueError:
        raise ValueError(
            f"Invalid date: {date_value}. "
            "Use YYYY-MM-DD format."
        )

In [10]:
def is_date_in_range(published_datetime, start_date, end_date):
    if published_datetime is None:
        return False

    article_date = published_datetime.date()

    return (
        start_date
        <= article_date
        <= end_date
    )

##Article Url Crawler

In [11]:
def crawl_article_urls(start_url=START_URL, max_pages=MAX_PAGES):
    pages_to_visit = [start_url]

    visited_pages = set()
    article_urls = []

    while (
        pages_to_visit
        and len(visited_pages) < max_pages
    ):
        current_url = pages_to_visit.pop(0)

        if current_url in visited_pages:
            continue

        visited_pages.add(current_url)

        print(
            f"Crawling page "
            f"{len(visited_pages)}/{max_pages}: "
            f"{current_url}"
        )

        soup = get_soup(current_url)

        if soup is None:
            continue

        for tag in soup.find_all(
            "a",
            href=True
        ):
            url = normalize_url(
                tag["href"],
                current_url
            )

            if not is_bisnis_url(url):
                continue

            if is_article_url(url):
                if url not in article_urls:
                    article_urls.append(url)

            else:
                if (
                    "page" in url.lower()
                    or url.count("/") <= 4
                ):
                    if (
                        url not in visited_pages
                        and url not in pages_to_visit
                    ):
                        pages_to_visit.append(url)

    return article_urls

## Global Scraping Config

In [12]:
def find_article_json(data):
    if isinstance(data, dict):
        article_type = data.get("@type", "")

        if isinstance(article_type, list):
            article_types = article_type
        else:
            article_types = [article_type]

        valid_types = {
            "Article",
            "NewsArticle",
            "ReportageNewsArticle",
            "AnalysisNewsArticle"
        }

        if valid_types.intersection(
            set(article_types)
        ):
            return data

        for value in data.values():
            result = find_article_json(value)

            if result:
                return result

    elif isinstance(data, list):
        for item in data:
            result = find_article_json(item)

            if result:
                return result

    return None

In [13]:
def get_json_ld(soup):
    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    for script in scripts:
        try:
            content = script.string

            if not content:
                content = script.get_text(
                    strip=True
                )

            if not content:
                continue

            data = json.loads(content)

            article_data = find_article_json(data)

            if article_data:
                return article_data

        except (
            json.JSONDecodeError,
            TypeError,
            AttributeError
        ):
            continue

    return {}

In [14]:
def get_meta(soup, property_name=None, name=None):
    if property_name:
        tag = soup.find(
            "meta",
            property=property_name
        )

    elif name:
        tag = soup.find(
            "meta",
            attrs={"name": name}
        )

    else:
        return None

    if not tag:
        return None

    content = tag.get("content")

    return content.strip() if content else None

In [15]:
def get_article_text(soup):
    selectors = [
        "[itemprop='articleBody']",
        ".article-body",
        ".detail-content",
        ".news-content",
        ".content-article",
        ".post-content",
        "article"
    ]

    for selector in selectors:
        container = soup.select_one(selector)

        if not container:
            continue

        paragraphs = []

        for paragraph in container.find_all("p"):
            text = paragraph.get_text(
                " ",
                strip=True
            )

            if len(text) >= 30:
                paragraphs.append(text)

        article_text = "\n\n".join(
            paragraphs
        )

        if article_text:
            return article_text

    return None

In [16]:
def scrape_article(url):
    soup = get_soup(url)

    if soup is None:
        return None

    json_ld = get_json_ld(soup)

    published_at = (
        json_ld.get("datePublished")
        or get_meta(
            soup,
            property_name="article:published_time"
        )
    )

    published_datetime = parse_date(
        published_at
    )

    title = (
        json_ld.get("headline")
        or get_meta(
            soup,
            property_name="og:title"
        )
    )

    article_text = (
        json_ld.get("articleBody")
        or get_article_text(soup)
    )

    if not title:
        print(f"Title not found: {url}")
        return None

    if published_datetime is None:
        print(f"Date not found: {url}")
        return None

    if not article_text:
        print(f"Content not found: {url}")
        return None

    return {
        "link": url,
        "judul": title,
        "isi_artikel": article_text,
        "tanggal_terbit": published_datetime.isoformat()
    }

## Import as JSON

In [17]:
def save_articles(articles, output_file):
    articles = sorted(
        articles,
        key=lambda article: article["tanggal_terbit"],
        reverse=True
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            articles,
            file,
            ensure_ascii=False,
            indent=4
        )

    print(
        f"\nSaved {len(articles)} articles "
        f"to {output_file}"
    )

#Backtrack Crawler

In [18]:
BACKTRACK_START_DATE = "2026-07-01"
BACKTRACK_END_DATE = "2026-07-10"

BACKTRACK_OUTPUT_FILE = (
    f"bisnis_articles_"
    f"{BACKTRACK_START_DATE}_to_"
    f"{BACKTRACK_END_DATE}.json"
)

In [19]:
def run_backtrack():
    start_date = parse_input_date(
        BACKTRACK_START_DATE
    )

    end_date = parse_input_date(
        BACKTRACK_END_DATE
    )

    if start_date > end_date:
        raise ValueError(
            "BACKTRACK_START_DATE cannot be "
            "later than BACKTRACK_END_DATE."
        )

    print("Starting Backtrack crawler.")

    print(
        f"Date range: "
        f"{BACKTRACK_START_DATE} to "
        f"{BACKTRACK_END_DATE}"
    )

    article_urls = crawl_article_urls()

    print(
        f"\nFound {len(article_urls)} "
        "possible article URLs."
    )

    articles = []

    for number, url in enumerate(
        article_urls,
        start=1
    ):
        print(
            f"\nScraping "
            f"{number}/{len(article_urls)}:"
        )
        print(url)

        article = scrape_article(url)

        if article is None:
            continue

        published_datetime = parse_date(
            article["tanggal_terbit"]
        )

        if is_date_in_range(
            published_datetime,
            start_date,
            end_date
        ):
            articles.append(article)

            print(
                f"Collected: "
                f"{article['tanggal_terbit']} - "
                f"{article['judul']}"
            )

        else:
            print(
                "Skipped: publication date "
                "is outside the range."
            )

    save_articles(
        articles,
        BACKTRACK_OUTPUT_FILE
    )

    return articles

In [20]:
run_backtrack()

Starting Backtrack crawler.
Date range: 2026-07-01 to 2026-07-10
Crawling page 1/5: https://www.bisnis.com/
Crawling page 2/5: https://premium.bisnis.com
Crawling page 3/5: https://epaper.bisnis.com
Crawling page 4/5: https://interaktif.bisnis.com
Crawling page 5/5: https://infografik.bisnis.com

Found 155 possible article URLs.

Scraping 1/155:
https://market.bisnis.com/read/20260723/7/1990508/ihsg-ditutup-melemah-030-ke-6315-saham-akra-bbca-asii-jadi-lesu
Skipped: publication date is outside the range.

Scraping 2/155:
https://market.bisnis.com/read/20260723/93/1990497/rupiah-ditutup-melemah-ke-rp17936-efek-lonjakan-harga-minyak-membayangi
Skipped: publication date is outside the range.

Scraping 3/155:
https://kabar24.bisnis.com/read/20260723/15/1990383/apa-itu-universitas-republik-indonesia-yang-dibentuk-prabowo-begini-seluk-beluknya
Skipped: publication date is outside the range.

Scraping 4/155:
https://ekonomi.bisnis.com/read/20260723/9/1990394/dari-sekotak-susu-tap-untuk-negeri

[{'link': 'https://infografik.bisnis.com/read/20260710/547/1987024/risiko-pusat-keuangan',
  'judul': 'Risiko Pusat Keuangan',
  'isi_artikel': 'Bisnis.com, JAKARTA — Rencana pembentukan Pusat Finansial Internasional Indonesia (PFII) dinilai dapat menjadi langkah strategis untuk memperkuat daya saing sektor keuangan nasional. Namun, di balik peluang tersebut, terdapat sejumlah risiko yang perlu diantisipasi agar kawasan ini benar-benar mampu menarik investasi baru tanpa menimbulkan dampak negatif terhadap perekonomian domestik.\n\nKajian Fakultas Ekonomi dan Bisnis Universitas Indonesia (FEB UI) mengingatkan bahwa desain tata kelola PFII menjadi faktor penentu keberhasilannya. Apabila tidak disusun secara cermat, kawasan tersebut berpotensi memunculkan praktik penyalahgunaan insentif, penghindaran pajak, hingga fenomena round tripping , yakni aliran modal domestik yang diputar ke luar negeri sebelum kembali masuk sebagai investasi asing.\n\nRisiko lain muncul dari rencana pemberian ins

In [21]:
from google.colab import files

files.download(
    BACKTRACK_OUTPUT_FILE
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Standard Crawler

In [22]:
STANDARD_MAX_ARTICLES = 10
STANDARD_INTERVAL_SECONDS = 30
STANDARD_MAX_CYCLES = 3
STANDARD_OUTPUT_FILE = "standard_articles.json"

seen_links = set()

In [23]:
def run_standard_once():
    print("\nStarting Standard crawler cycle")

    article_urls = crawl_article_urls()

    print(
        f"\nFound {len(article_urls)} "
        "possible article URLs."
    )

    articles = []

    for number, url in enumerate(
        article_urls,
        start=1
    ):
        print(
            f"\nChecking "
            f"{number}/{len(article_urls)}:"
        )
        print(url)

        if url in seen_links:
            print(
                "Skipped: article already processed."
            )
            continue

        article = scrape_article(url)

        if article is None:
            continue

        articles.append(article)

        seen_links.add(url)

        print(
            f"Collected: "
            f"{article['tanggal_terbit']} - "
            f"{article['judul']}"
        )

        if len(articles) >= STANDARD_MAX_ARTICLES:
            print(
                f"\nReached the limit of "
                f"{STANDARD_MAX_ARTICLES} "
                "new articles."
            )
            break

    if articles:
        save_articles(
            articles,
            STANDARD_OUTPUT_FILE
        )

        print(
            f"\nSaved {len(articles)} "
            "new articles."
        )
    else:
        print("\nNo new articles found.")

    return articles

In [24]:
def run_standard():
    print("Starting Standard crawler.")

    for cycle in range(
        1,
        STANDARD_MAX_CYCLES + 1
    ):
        print(
            f"\nCycle {cycle}/"
            f"{STANDARD_MAX_CYCLES}"
        )

        run_standard_once()

        if cycle < STANDARD_MAX_CYCLES:
            print(
                f"\nWaiting "
                f"{STANDARD_INTERVAL_SECONDS} "
                "seconds before the next cycle"
            )

            time.sleep(
                STANDARD_INTERVAL_SECONDS
            )

    print(
        f"\nStandard crawler finished "
        f"after {STANDARD_MAX_CYCLES} cycles."
    )

In [25]:
run_standard()

Starting Standard crawler.

Cycle 1/3

Starting Standard crawler cycle
Crawling page 1/5: https://www.bisnis.com/
Crawling page 2/5: https://premium.bisnis.com
Crawling page 3/5: https://epaper.bisnis.com
Crawling page 4/5: https://interaktif.bisnis.com
Crawling page 5/5: https://infografik.bisnis.com

Found 155 possible article URLs.

Checking 1/155:
https://market.bisnis.com/read/20260723/7/1990508/ihsg-ditutup-melemah-030-ke-6315-saham-akra-bbca-asii-jadi-lesu
Collected: 2026-07-23T16:17:25 - IHSG Ditutup Melemah 0,30% ke 6.315, Saham AKRA, BBCA & ASII Jadi Lesu

Checking 2/155:
https://market.bisnis.com/read/20260723/93/1990497/rupiah-ditutup-melemah-ke-rp17936-efek-lonjakan-harga-minyak-membayangi
Collected: 2026-07-23T15:31:14 - Rupiah Ditutup Melemah ke Rp17.936 Efek Lonjakan Harga Minyak Membayangi

Checking 3/155:
https://kabar24.bisnis.com/read/20260723/15/1990383/apa-itu-universitas-republik-indonesia-yang-dibentuk-prabowo-begini-seluk-beluknya
Collected: 2026-07-23T10:31:21

In [26]:
from google.colab import files

files.download(
    STANDARD_OUTPUT_FILE
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>